In [1]:
from flask import Flask, render_template, request, jsonify
from torchvision import transforms
from PIL import Image
import torch
import torch.nn.functional as F

app = Flask(__name__)

# Models
vgg16 = torch.load('vgg16_model.pth', map_location=torch.device('cpu'))
vgg16.eval()

vit = torch.load('vit_model.pth', map_location=torch.device('cpu'))
vit.eval()

# Transformations
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Emotions
class_labels = [
    'Angry',
    'Disgusted',
    'Fearful',
    'Happy',
    'Neutral',
    'Sad',
    'Surprised'
]

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/classify', methods=['POST'])
def classify():
    # Get the image from the POST request
    try:
        img_file = request.files['image']

        # Process the image
        img = Image.open(img_file.stream)
        img = transform(img)
        img = img.unsqueeze(0)

        selected_model = request.form['model']

        # Select the model to classify with
        if selected_model == 'vgg16':
            model = vgg16
        elif selected_model == 'vit':
            model = vit
        else:
            return 'Invalid model selected'

        # Predict the emotion
        with torch.no_grad():
            output = model(img)
            probabilities = F.softmax(output, dim=1)[0]
            confidence, predicted_class = torch.max(probabilities, 0)

            class_probabilities = {class_labels[i]: 
                                   round(float(probabilities[i]*100),2) 
                                   for i in range(len(class_labels))}

            result = {'class': class_labels[predicted_class],
                      'confidence': round(float(confidence)*100,2),
                      'probabilities': class_probabilities}

        return render_template('index.html', result=result)

    except Exception as e:
        error = "Something went wrong. Try submitting a different image."
        return render_template('index.html', error=error)

if __name__ == '__main__':
    app.run()

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL torchvision.models.vgg.VGG was not an allowed global by default. Please use `torch.serialization.add_safe_globals([torchvision.models.vgg.VGG])` or the `torch.serialization.safe_globals([torchvision.models.vgg.VGG])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more abou